# 05 – Spatial Extrapolation and Multi-Year Time Series

Reproduces Sections 5.3 and 5.4:
- Apply trained QRF models to all SAR/Sentinel-2 composites across the
  study extent → wall-to-wall predictions of structural metrics
- Produce annual time series (2019–2024) at the 20 m pixel level
- Prediction intervals (10th/50th/90th quantiles) reflecting model
  uncertainty (Section 5.2)
- Plot and save outputs as GeoTIFFs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config
from src.lidar_metrics import METRIC_NAMES

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import joblib
import matplotlib.pyplot as plt
from src.remote_sensing import build_predictor_stack

## 5.1 Load composites and models

In [ ]:
# sar_all = xr.open_zarr(config.SAR_ZARR)
# s2_all = xr.open_zarr(config.S2_ZARR)
sar_all = xr.open_dataset(config.SAR_NC)
s2_all = xr.open_dataset(config.S2_NC)

model_dir = config.OUTPUT_DIR / 'models'
# models = {
#     m: joblib.load(model_dir / f'qrf_{m.replace(">", "gt").replace("<", "lt")}.joblib')
#     for m in METRIC_NAMES
#     if (model_dir / f'qrf_{m.replace(">", "gt").replace("<", "lt")}.joblib').exists()
# }
models = {
    m: joblib.load(
        model_dir / f'{config.LAS_NAME}qrf_{m.replace(">", "gt").replace("<", "lt")}.joblib'
    )
    for m in METRIC_NAMES
    if (
        model_dir / f'{config.LAS_NAME}qrf_{m.replace(">", "gt").replace("<", "lt")}.joblib'
    ).exists()
}

print(f"Loaded {len(models)} QRF models")
print("Available periods:", list(sar_all.period.values))

## 5.2 Build predictor stack for a given period

## 5.3 Predict structural metrics across the full extent

In [ ]:
QUANTILES = [0.1, 0.5, 0.9]   # 10th, median, 90th (Section 5.2)
PREDICT_METRICS = ['GFP', 'PAI', 'stdev', 'p50']   # subset for speed; remove to run all
PREDICT_PERIOD = 'dryseason'   # or 'annual_2022' etc.

X, valid_mask, feature_names, meta = build_predictor_stack(
    sar_all, s2_all, PREDICT_PERIOD
)
print(f"Prediction grid: {meta['height']} × {meta['width']} = {meta['height']*meta['width']:,} pixels")
print(f"Valid pixels: {valid_mask.sum():,}")

X_valid = X[valid_mask]

predictions = {}   # metric → (n_pixels, 3) array [q10, q50, q90]
for metric in PREDICT_METRICS:
    if metric not in models:
        continue
    model = models[metric]
    preds_valid = model.predict(X_valid, quantiles=QUANTILES)  # (n_valid, 3)
    full = np.full((len(valid_mask), 3), np.nan, dtype=np.float32)
    full[valid_mask] = preds_valid
    predictions[metric] = full.reshape(meta['height'], meta['width'], 3)
    print(f"  {metric}: done")

## 5.4 Write prediction GeoTIFFs (q10, q50, q90)

In [ ]:
from pyproj import CRS as ProjCRS

dx = float(meta['x'][1] - meta['x'][0])
dy = float(meta['y'][1] - meta['y'][0])
xmin = float(meta['x'][0]) - dx / 2
ymax = float(meta['y'][0]) - dy / 2
transform = from_origin(xmin, ymax, abs(dx), abs(dy))

for metric, pred_cube in predictions.items():
    out_path = config.OUTPUT_DIR / (config.LAS_NAME+f'_predict_{PREDICT_PERIOD}_{metric}.tif')
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=meta['height'], width=meta['width'],
        count=3, dtype='float32',
        crs=config.TARGET_CRS, transform=transform, nodata=np.nan,
    ) as dst:
        for qi, qname in enumerate(['q10', 'q50', 'q90']):
            dst.write(pred_cube[:, :, qi], qi + 1)
            dst.set_band_description(qi + 1, f'{metric}_{qname}')
    print(f"Written: {out_path}")

## 5.5 Annual time series (one prediction per year)

In [ ]:
annual_periods = [p for p in sar_all.period.values if str(p).startswith('annual_')]
print("Annual periods available:", annual_periods)

ts_metric = 'GFP'    # change as needed
ts_data = {}         # period → median prediction array

for period in annual_periods:
    if period not in [str(p) for p in sar_all.period.values]:
        continue
    X_yr, vm_yr, _, meta_yr = build_predictor_stack(sar_all, s2_all, period)
    if not vm_yr.any():
        continue
    model = models[ts_metric]
    preds = model.predict(X_yr[vm_yr], quantiles=0.5)
    full = np.full(len(vm_yr), np.nan, dtype=np.float32)
    full[vm_yr] = preds
    ts_data[period] = full.reshape(meta_yr['height'], meta_yr['width'])

print(f"Annual predictions for {ts_metric}: {list(ts_data.keys())}")

In [ ]:
# Spatial mean ± std time series (Section 5.4, Figure 16 equivalent)
years_plot = sorted(ts_data.keys())
means = [np.nanmean(ts_data[yr]) for yr in years_plot]
stds = [np.nanstd(ts_data[yr]) for yr in years_plot]
year_labels = [str(yr).replace('annual_', '') for yr in years_plot]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(year_labels, means, 'o-', color='steelblue', linewidth=2, label='Spatial mean')
ax.fill_between(year_labels,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.2, color='steelblue', label='±1 SD')
ax.set_xlabel('Year')
ax.set_ylabel(ts_metric)
ax.set_title(f'Annual time series: {ts_metric} (spatial mean ± SD)')
ax.legend()
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / (config.LAS_NAME+f'_fig_timeseries_{ts_metric}.png'), dpi=150)
plt.show()

In [ ]:
# Spatial maps for each year side-by-side
n = len(ts_data)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
for ax, yr in zip(np.atleast_1d(axes), sorted(ts_data.keys())):
    arr = ts_data[yr]
    im = ax.imshow(arr, cmap='Greens_r', origin='upper',
                   vmin=np.nanpercentile(arr, 5), vmax=np.nanpercentile(arr, 95))
    ax.set_title(str(yr).replace('annual_', ''))
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8, label=ts_metric)
plt.suptitle(f'Annual {ts_metric} maps (QRF median prediction)', y=1.01)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / (config.LAS_NAME+f'_fig_annual_maps_{ts_metric}.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.6 Uncertainty maps (Q90 − Q10 prediction interval width)

In [ ]:
# Write and plot uncertainty for each predicted metric
for metric, pred_cube in predictions.items():
    interval_width = pred_cube[:, :, 2] - pred_cube[:, :, 0]   # Q90 - Q10
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(interval_width, cmap='YlOrRd', origin='upper')
    ax.set_title(f'{metric} prediction interval width (Q90 – Q10)')
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='Q90 – Q10')
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / (config.LAS_NAME+f'_fig_uncertainty_{metric}.png'), dpi=150)
    plt.show()